In [1]:
import pandas as pd
import numpy as np

data = pd.read_csv('../data/sp500_raw.csv', index_col=0, parse_dates=True)
data['log_return'] = np.log(data['Close'] / data['Close'].shift(1))
data = data.dropna()
data.head()

,Close,High,Low,Open,Volume,log_return
Date,,,,,,
2015-01-05,2020.579956,2054.439941,2017.339966,2054.439941,3799120000,-0.018447
2015-01-06,2002.609985,2030.250000,1992.439941,2022.150024,4460110000,-0.008933
2015-01-07,2025.900024,2029.609985,2005.550049,2005.550049,3805480000,0.011563
2015-01-08,2062.139893,2064.080078,2030.609985,2030.609985,3934010000,0.017730
2015-01-09,2044.810059,2064.429932,2038.329956,2063.449951,3364140000,-0.008439


In [2]:
data['realized_volatility_5'] = data['log_return'].rolling(5).std() * np.sqrt(252)
data['realized_volatility_18'] = data['log_return'].rolling(18).std() * np.sqrt(252)
data['realized_volatility_21'] = data['log_return'].rolling(21).std() * np.sqrt(252)
data.head(5)

,Close,High,Low,Open,Volume,log_return,realized_volatility_5,realized_volatility_18,realized_volatility_21
Date,,,,,,,,,
2015-01-05,2020.579956,2054.439941,2017.339966,2054.439941,3799120000,-0.018447,NaN,NaN,NaN
2015-01-06,2002.609985,2030.250000,1992.439941,2022.150024,4460110000,-0.008933,NaN,NaN,NaN
2015-01-07,2025.900024,2029.609985,2005.550049,2005.550049,3805480000,0.011563,NaN,NaN,NaN
2015-01-08,2062.139893,2064.080078,2030.609985,2030.609985,3934010000,0.017730,NaN,NaN,NaN
2015-01-09,2044.810059,2064.429932,2038.329956,2063.449951,3364140000,-0.008439,0.242166,NaN,NaN


In [3]:
data['return_lag_1'] = data['log_return'].shift(1)
data['return_lag_2'] = data['log_return'].shift(2)
data['return_lag_3'] = data['log_return'].shift(3)
data['return_lag_5'] = data['log_return'].shift(5)

In [4]:
data['volume_change'] = data['Volume'].pct_change()
data['volume_ma_5'] = data['Volume'].rolling(5).mean()

In [5]:
data['day_of_week'] = data.index.dayofweek
data['month'] = data.index.month

In [6]:
data['future_volatility_21'] = data['log_return'].rolling(21).std().shift(-21) * np.sqrt(252)
data = data.dropna()
data[['realized_volatility_21', 'future_volatility_21']].head(10)

,realized_volatility_21,future_volatility_21
Date,,
2015-02-03,0.180978,0.078070
2015-02-04,0.169203,0.093172
2015-02-05,0.168782,0.087412
2015-02-06,0.165366,0.105841
2015-02-09,0.153966,0.105036
2015-02-10,0.155507,0.107670
2015-02-11,0.152311,0.109678
2015-02-12,0.154593,0.114762
2015-02-13,0.152454,0.114182


In [7]:
feature_cols = ['realized_volatility_5', 'realized_volatility_18', 'realized_volatility_21',
                'return_lag_1', 'return_lag_2', 'return_lag_3', 'return_lag_5',
                'volume_change', 'volume_ma_5', 'day_of_week', 'month']

correlations = data[feature_cols + ['future_volatility_21']].corr()['future_volatility_21'].sort_values(ascending=False)
print(correlations)

future_volatility_21      1.000000
realized_volatility_5     0.534648
realized_volatility_18    0.504820
realized_volatility_21    0.490044
volume_ma_5               0.409912
volume_change             0.018582
day_of_week               0.002149
return_lag_5             -0.091187
return_lag_3             -0.109788
return_lag_2             -0.117891
return_lag_1             -0.129016
month                    -0.162157
Name: future_volatility_21, dtype: float64


In [8]:
data.to_csv('../data/sp500_features.csv')
print(f"Dataset shape: {data.shape}")
print(f"Features: {feature_cols}")

Dataset shape: (2724, 18)
Features: ['realized_volatility_5', 'realized_volatility_18', 'realized_volatility_21', 'return_lag_1', 'return_lag_2', 'return_lag_3', 'return_lag_5', 'volume_change', 'volume_ma_5', 'day_of_week', 'month']


## Summary

Engineered a feature set to predict future realized volatility, using only information
available at each point in time (no look-ahead bias):

- **Realized volatility** at 5, 18, and 21-day windows (annualized)
- **Lagged returns** (1, 2, 3, 5 days back)
- **Volume features**: percentage change and 5-day moving average
- **Calendar features**: day of week, month

**Target:** realized volatility over the *next* 21 trading days, computed by shifting a
forward-looking rolling window (`shift(-21)`), ensuring no data leakage from future observations
into the features.

### Feature correlation with target (future 21-day volatility)

| Feature | Correlation |
|---|---|
| realized_volatility_5 | 0.535 |
| realized_volatility_18 | 0.505 |
| realized_volatility_21 | 0.490 |
| volume_ma_5 | 0.410 |
| volume_change | 0.019 |
| day_of_week | 0.002 |
| return_lag_5 | -0.091 |
| return_lag_3 | -0.110 |
| return_lag_2 | -0.118 |
| return_lag_1 | -0.129 |
| month | -0.162 |

**Key finding:** Short-term realized volatility (5-day window) is the strongest predictor of
future volatility, consistent with the volatility clustering confirmed by the Ljung-Box test in
Day 1 — recent volatility carries more predictive signal than longer historical windows.
Volume (5-day average) is also meaningfully predictive. Individual lagged returns show a weak
but consistent negative correlation, suggesting a mild mean-reversion effect in volatility
following large single-day moves. Day-of-week has no linear relationship with future volatility;
volume_change is likewise uninformative in this linear analysis.

Final feature set for modeling: `realized_volatility_5`, `realized_volatility_18`,
`realized_volatility_21`, `volume_ma_5`, `month` (weak calendar effect retained).
`return_lag_1` through `return_lag_5` were dropped despite showing a consistent negative
correlation — the pattern is monotonic (lag_1 strongest, lag_5 weakest), suggesting these
lags mostly duplicate the mean-reversion signal already captured more directly by the
realized volatility features. `day_of_week` and `volume_change` were dropped as uninformative
(correlation ≈ 0).